# Gas-turbine combustor (reacting mean flow)

A complete combustor as a Nefes network: **compressor-discharge air** -> a
**fuel mass source** (the injector) -> an **equilibrium flame** (ignition) ->
a **dilution-air mass source** (cooling to the turbine-inlet temperature) ->
the **turbine-inlet** outlet.

![Network topology](gas_turbine_combustor_topology.png)

Key ideas exercised here:

* the thermo data is read from the NASA Glenn / CEA `thermo.inp` database via
  the bundled `nefes.thermo`; you pick the species you need;
* streams are specified by **species** name (`{'O2': 0.21, 'N2': 0.79}`); the
  network transports one conserved **mixture fraction** per distinct injected
  composition (here: air and fuel), discovered automatically at build time;
* the **mass source** adds mass / momentum / energy with the proper source
  terms -- a fuel injector is just a mass source that injects fuel;
* **ignition** is done only by the flame element (frozen unburnt -> equilibrium
  burnt); the injector never reacts.

Tweak `mdot_fuel`, `mdot_dilution`, `Tair`, `p` in the builder and re-run.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import nefes
from nefes.elements import catalog as cat
from nefes.thermo import ThermoInp, EQ_FROZEN, EQ_KERNEL
from nefes.chem import resolve_composition, enthalpy_mass
from nefes.plotting import use_nefes_theme, COLORWAY

# Use the bundled theme instead of plotly's default.
use_nefes_theme()

## Species library
A small CH4/air library selected from `thermo.inp`: the air species and the
fuel, plus the major combustion products the equilibrium solver fills in.

In [ ]:
lib = ThermoInp().library(
    ["O2", "N2", "Ar", "CO2", "CH4", "H2O", "CO", "OH", "H", "O", "NO", "H2"]
)
print("elements:", lib.elements)
print("species :", [s.name for s in lib.species])

## Build the combustor network
The builder assembles the five elements and the four edges into a `Network`.
`Network.solve()` seeds itself by **propagating the feeds through the network**
(mass-weighted mixing of each stream's `(mdot, h_t, xi)`), so no hand-built
per-edge guess is needed -- even though the network spans a wide enthalpy range
(cold air vs hot products vs CH4's strongly negative formation enthalpy).

In [ ]:
AIR = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
FUEL = {"CH4": 1.0}


def build_combustor(mdot_air=1.0, mdot_fuel=0.045, mdot_dil=1.0, Tair=700.0, p=10.0e5, A=0.05):
    """air-in -> fuel injector -> flame -> dilution -> turbine-in.

    Air and fuel are the two feed streams (the dilution injects the same air, so it
    shares the air mixture fraction).  They are discovered from the element
    compositions at build time; ``Network.solve()`` then seeds every edge by
    propagating the feeds through the network -- no hand-built guess.
    """
    Yair, _ = resolve_composition(lib, AIR)
    Yfuel, _ = resolve_composition(lib, FUEL)
    h_air = enthalpy_mass(lib, Yair, Tair)
    h_fuel = enthalpy_mass(lib, Yfuel, Tair)
    m1 = mdot_air + mdot_fuel  # after the injector
    m2 = m1 + mdot_dil  # after dilution
    h_mix = (mdot_air * h_air + mdot_fuel * h_fuel) / m1  # enthalpy scale for h_ref

    nodes = [
        cat.mass_flow_inlet(mdot_air, Tair, composition=AIR, name="air-in"),
        cat.mass_source(mdot_fuel, Tair, composition=FUEL, name="fuel"),
        cat.equilibrium_flame(name="flame"),
        cat.mass_source(mdot_dil, Tair, composition=AIR, name="dilution"),
        cat.pressure_outlet(p, Tt_backflow=Tair, composition=AIR, name="turbine-in"),
    ]
    edges = [(0, 1, A), (1, 2, A), (2, 3, A), (3, 4, A)]
    # Pin the closures explicitly: the two post-flame edges stay at equilibrium (EQ_KERNEL).  The
    # dilution injects inert air (a fresh, marker-0 stream) into the burnt gas, which would pull the
    # automatic burnt-marker below 1 on the turbine-inlet edge and run it as a frozen/equilibrium
    # blend -- so the explicit per-edge closure is the right tool here, not marker gating.
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
    return nefes.Network(
        nefes.equilibrium(lib), nodes=nodes, edges=edges, edge_models=edge_models,
    )


net = build_combustor()
sol = net.solve()
print("converged:", sol.converged, " Newton iterations:", sol.iterations)

## Converged edge states

In [ ]:
names = ["air", "air+fuel", "burnt", "diluted"]
print(f"{'edge':<10}{'mdot':>9}{'T [K]':>10}{'p [bar]':>10}{'M':>8}{'h_t [J/kg]':>14}")
for e, label in enumerate(names):
    st = sol.edge(e)
    print(f"{label:<10}{st['mdot']:9.4f}{st['T']:10.1f}"
          f"{st['p'] / 1e5:10.3f}{st['M']:8.3f}{st['h_t']:14.3e}")

print()
print(f"global mass in  : {sol.edge(0)['mdot']:.4f} kg/s (air) + injected fuel + dilution")
print(f"global mass out : {sol.edge(3)['mdot']:.4f} kg/s")
print(f"flame conserves h_t: {sol.edge(1)['h_t']:.4e} -> {sol.edge(2)['h_t']:.4e} J/kg")

## Axial temperature profile
The flame jump, then dilution cooling to the turbine-inlet temperature.

In [ ]:
names = ["air", "air+fuel", "burnt", "diluted"]
x_ax = [0, 1, 2, 3]
stations = ["air-in", "injector", "flame", "turbine-in"]
T = sol.field("T")
fig = go.Figure()
fig.add_scatter(x=x_ax, y=T, mode="lines+markers+text",
                line=dict(color=COLORWAY[4], width=2), marker=dict(size=10),
                text=[f"{nm}<br>{ti:.0f} K" for nm, ti in zip(names, T)],
                textposition="top center", cliponaxis=False)
fig.update_layout(title="Combustor axial temperature",
                  yaxis_title="static temperature  [K]", showlegend=False, height=440)
fig.update_xaxes(tickmode="array", tickvals=x_ax, ticktext=stations)
fig.show()

## Fuel-flow sweep: equivalence ratio -> flame & turbine-inlet temperature
With the dilution flow fixed, raising the fuel flow raises the primary flame
temperature and the diluted turbine-inlet temperature.

In [ ]:
# stoichiometric CH4/air mass ratio (for the equivalence ratio of the primary zone)
f_stoich = 0.05518
fuels = np.linspace(0.02, 0.07, 11)
Tflame, Tturb, phi = [], [], []
prev = None
for mf in fuels:
    # Warm-start each point from the last converged state; a parameter write to the
    # fuel mass flow never changes the network layout.
    s = net.with_params({"fuel.mdot": float(mf)}).solve(x0=prev.x if prev else None)
    prev = s
    Tflame.append(s.field("T")[2])
    Tturb.append(s.field("T")[3])
    phi.append((mf / 1.0) / f_stoich)

fig = go.Figure()
fig.add_scatter(x=phi, y=Tflame, mode="lines+markers", name="flame (primary)", line_color=COLORWAY[4])
fig.add_scatter(x=phi, y=Tturb, mode="lines+markers", name="turbine-inlet (diluted)", line_color=COLORWAY[0])
fig.update_layout(title="Fuel-flow sweep",
                  xaxis_title="primary equivalence ratio  φ", yaxis_title="temperature  [K]", height=440)
fig.show()

## Dilution sweep: trading dilution air for turbine-inlet temperature
More dilution air cools the products toward the metallurgical turbine-inlet limit.

In [ ]:
dils = np.linspace(0.3, 2.0, 11)
Tturb_d = []
prev = None
for md in dils:
    s = net.with_params({"dilution.mdot": float(md)}).solve(x0=prev.x if prev else None)
    prev = s
    Tturb_d.append(s.field("T")[3])

fig = go.Figure()
fig.add_scatter(x=dils, y=Tturb_d, mode="lines+markers", line_color=COLORWAY[2])
fig.update_layout(title="Dilution sweep",
                  xaxis_title="dilution air ṁ  [kg/s]", yaxis_title="turbine-inlet T  [K]",
                  showlegend=False, height=440)
fig.show()

## Export for Nemo

The full network is available as `net` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML -- `sol.to_yaml(path)` embeds the mean-flow result fields, `net.to_yaml(path)` writes the topology only -- then open the file in Nemo.

In [ ]:
import os, tempfile

_out = os.path.join(tempfile.mkdtemp(), "gas_turbine_combustor.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use net.to_yaml(_out) for topology only
print("saved case:", _out)